In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/object-detetction-week-10-dlp/sample_submission.csv
/kaggle/input/competitions/object-detetction-week-10-dlp/train.csv
/kaggle/input/competitions/object-detetction-week-10-dlp/test/0040296.jpg
/kaggle/input/competitions/object-detetction-week-10-dlp/test/0015674.jpg
/kaggle/input/competitions/object-detetction-week-10-dlp/test/0033897.jpg
/kaggle/input/competitions/object-detetction-week-10-dlp/test/0020075.jpg
/kaggle/input/competitions/object-detetction-week-10-dlp/test/0004880.jpg
/kaggle/input/competitions/object-detetction-week-10-dlp/test/0023040.jpg
/kaggle/input/competitions/object-detetction-week-10-dlp/test/0038189.jpg
/kaggle/input/competitions/object-detetction-week-10-dlp/test/0018427.jpg
/kaggle/input/competitions/object-detetction-week-10-dlp/test/0010360.jpg
/kaggle/input/competitions/object-detetction-week-10-dlp/test/0024333.jpg
/kaggle/input/competitions/object-detetction-week-10-dlp/test/007939.jpg
/kaggle/input/competitions/object-detetct

In [2]:
TRAIN_IMG_DIR = "/kaggle/input/competitions/object-detetction-week-10-dlp/train"
TEST_IMG_DIR  = "/kaggle/input/competitions/object-detetction-week-10-dlp/test"
CSV_PATH      = "/kaggle/input/competitions/object-detetction-week-10-dlp/train.csv"

In [3]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

print(df.columns)
df.head()

Index(['image_id', 'class_name', 'x_min', 'y_min', 'x_max', 'y_max', 'width',
       'height'],
      dtype='object')


,image_id,class_name,x_min,y_min,x_max,y_max,width,height
0,0000001.jpg,RiceLeafRoller,241,52,295,93,800,600
1,0000001.jpg,RiceLeafRoller,242,95,283,142,800,600
2,0000001.jpg,RiceLeafCaterpillar,157,379,185,410,800,600
3,0000002.jpg,PaddyStemMaggot,323,215,414,281,800,600
4,0000004.jpg,YellowRiceBorer,523,435,542,472,800,600


In [4]:
import os
import cv2
from tqdm import tqdm

LABEL_DIR = "/kaggle/working/labels"
os.makedirs(LABEL_DIR, exist_ok=True)

classes = sorted(df['class_name'].unique())
class_to_id = {c:i for i,c in enumerate(classes)}

for img_id, group in tqdm(df.groupby("image_id")):
    img_path = os.path.join(TRAIN_IMG_DIR, img_id)

    img = cv2.imread(img_path)
    h, w, _ = img.shape

    label_file = os.path.join(LABEL_DIR, img_id.replace(".jpg", ".txt"))

    with open(label_file, "w") as f:
        for _, row in group.iterrows():
            cls = class_to_id[row['class_name']]

            x1, y1, x2, y2 = row['x_min'], row['y_min'], row['x_max'], row['y_max']

            # YOLO conversion
            xc = ((x1 + x2) / 2) / w
            yc = ((y1 + y2) / 2) / h
            bw = (x2 - x1) / w
            bh = (y2 - y1) / h

            f.write(f"{cls} {xc} {yc} {bw} {bh}\n")

100%|██████████| 12701/12701 [01:52<00:00, 112.84it/s]


In [5]:
BASE = "/kaggle/working/dataset"

for split in ["train", "val"]:
    os.makedirs(f"{BASE}/images/{split}", exist_ok=True)
    os.makedirs(f"{BASE}/labels/{split}", exist_ok=True)

In [6]:
from sklearn.model_selection import train_test_split
import shutil

images = df['image_id'].unique()

train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

def move(img_list, split):
    for img in img_list:
        shutil.copy(f"{TRAIN_IMG_DIR}/{img}", f"{BASE}/images/{split}/{img}")
        shutil.copy(f"{LABEL_DIR}/{img.replace('.jpg','.txt')}",
                    f"{BASE}/labels/{split}/{img.replace('.jpg','.txt')}")

move(train_imgs, "train")
move(val_imgs, "val")

In [7]:
yaml_text = f"""
path: {BASE}

train: images/train
val: images/val

names:
"""

for i, c in enumerate(classes):
    yaml_text += f"  {i}: {c}\n"

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(yaml_text)

In [8]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.4 MB/s eta 0:00:00


In [9]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data="/kaggle/working/data.yaml",
    epochs=30,
    imgsz=1024,
    batch=8
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.39 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b07c7f1f3b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.0420

In [10]:
results = model.predict(
    source=TEST_IMG_DIR,
    imgsz=1024,
    conf=0.25
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/7600 /kaggle/input/competitions/object-detetction-week-10-dlp/test/0000003.jpg: 768x1024 1 AsiaticRiceBorer, 2 BrownPlantHoppers, 2 RiceLeafCaterpillars, 3 RiceStemflys, 1 YellowRiceBorer, 56.0ms
image 2/7600 /kaggle/input/competitions/object-detetction-week-10-dlp/test/0000010.jpg: 768x1024 1 PaddyStemMaggot, 20 YellowRiceBorers, 52.0ms
image 3/7600 /kaggle/input/competitions/object-detetction-week-10-dlp/test/0000012.jpg: 768x1024 1 RiceLeafCat

In [11]:
import pandas as pd
import os

id_to_class = {v: k for k, v in class_to_id.items()}

rows = []

DUMMY_PRED = "no_object 1 0 0 0 0"   # <-- important

for r in results:
    name = os.path.basename(r.path)
    preds = []

    if r.boxes is not None and len(r.boxes) > 0:
        for box, conf, cls in zip(r.boxes.xyxy, r.boxes.conf, r.boxes.cls):
            x1, y1, x2, y2 = map(int, box)
            label = id_to_class[int(cls)]

            preds.append(f"{label} {conf:.4f} {x1} {y1} {x2} {y2}")

    if preds:
        prediction_string = " ".join(preds)
    else:
        prediction_string = DUMMY_PRED

    rows.append({
        "ImageID": name,
        "PredictionString": prediction_string
    })

df = pd.DataFrame(rows)

df["PredictionString"] = df["PredictionString"].replace("", DUMMY_PRED)

df.to_csv("submission.csv", index=False)